# 05 ? Classifier Two-Sample Test (C2ST) mechanism probe

This is a one-shot exploratory probe, not a confirmation exercise. Notebook 02's ranking result stands regardless of this outcome: C2ST asks only whether a SMOTE-synthetic minority customer is distinguishable from a real minority customer in the flattened honest-window panel feature space.

AUC near 0.5 is a valid result: it means point-level synthetic rows are hard to distinguish from real rows under this probe, and the harm mechanism likely lives elsewhere. AUC meaningfully above 0.5 means the classifier can detect synthetic rows, and permutation importances help locate where that detectability concentrates.

All reusable logic lives in `src/evaluation/c2st.py`; this notebook only orchestrates calls, displays results, and writes the verification report.

## Setup

In [1]:
import hashlib
import inspect
import json
import platform
import sys
from pathlib import Path

import imblearn
import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "README.md").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loaders import SIGNALS, load_churn_panel
from src.evaluation.c2st import (
    build_real_vs_duplicated,
    build_real_vs_synthetic,
    c2st_auc,
    c2st_importances,
)
import src.evaluation.c2st as c2st_module
from src.features.windows import WINDOW_PRESETS
from src.utils.config import OUTPUT_DIR, RAW_CHURN_PATH, SEED, set_global_seed

set_global_seed()

N_SPLITS = 5
N_REPEATS = 3
HONEST_MONTHS = WINDOW_PRESETS["HONEST"]
SIGNALS_USED = SIGNALS

df = load_churn_panel()
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
print(f"Honest window: {', '.join(HONEST_MONTHS)}")
print(f"Signals: {', '.join(SIGNALS_USED)}")

Loaded 99,807 rows x 55 columns
Honest window: N-11, N-10, N-9, N-8, N-7, N-6, N-5, N-4, N-3, N-2
Signals: TAG, STATUS_BAYAR, GGN, USAGE


## 1. Build C2ST datasets

The real-vs-SMOTE task uses real churner rows and a balanced sample of newly synthesized SMOTE minority rows. The real-vs-duplicated task uses real churner rows and same-size random duplicated churner rows; this is the sanity floor because copies of real rows should be indistinguishable from real rows.

In [2]:
X_smote, y_smote = build_real_vs_synthetic(
    df, HONEST_MONTHS, signals=SIGNALS_USED, seed=SEED
)
X_dup, y_dup = build_real_vs_duplicated(df, HONEST_MONTHS, signals=SIGNALS_USED, seed=SEED)

metadata = {
    "real_vs_smote": {k: v for k, v in X_smote.attrs.items() if not k.startswith("_")},
    "real_vs_duplicated": {k: v for k, v in X_dup.attrs.items() if not k.startswith("_")},
}

print("real-vs-SMOTE metadata:")
print(json.dumps(metadata["real_vs_smote"], indent=2))
print("\nreal-vs-duplicated metadata:")
print(json.dumps(metadata["real_vs_duplicated"], indent=2))
print(f"\nX_smote shape: {X_smote.shape}, positive_rate={y_smote.mean():.6f}")
print(f"X_dup shape:   {X_dup.shape}, positive_rate={y_dup.mean():.6f}")

real-vs-SMOTE metadata:
{
  "task": "real_vs_smote",
  "n_real": 3804,
  "n_smote_synthetic_total": 92198,
  "n_smote_synthetic_used": 3804,
  "n_features": 40,
  "n_dropped_missing": 1,
  "positive_rate": 0.5,
  "smote_k_neighbors": 5,
  "months": [
    "N-11",
    "N-10",
    "N-9",
    "N-8",
    "N-7",
    "N-6",
    "N-5",
    "N-4",
    "N-3",
    "N-2"
  ],
  "signals": [
    "TAG",
    "STATUS_BAYAR",
    "GGN",
    "USAGE"
  ],
  "log1p_signals": [
    "TAG",
    "USAGE"
  ]
}

real-vs-duplicated metadata:
{
  "task": "real_vs_duplicated",
  "n_real": 3804,
  "n_duplicated": 3804,
  "n_features": 40,
  "n_dropped_missing": 1,
  "positive_rate": 0.5,
  "duplicate_policy": "one_exact_copy_per_real_row",
  "seed": 42,
  "months": [
    "N-11",
    "N-10",
    "N-9",
    "N-8",
    "N-7",
    "N-6",
    "N-5",
    "N-4",
    "N-3",
    "N-2"
  ],
  "signals": [
    "TAG",
    "STATUS_BAYAR",
    "GGN",
    "USAGE"
  ],
  "log1p_signals": [
    "TAG",
    "USAGE"
  ]
}

X_smote sha

## 2. C2ST AUC

Rule of thumb: approximately 0.50?0.55 is indistinguishable, 0.55?0.70 is mildly detectable, and above 0.70 is clearly detectable.

In [3]:
smote_auc = c2st_auc(X_smote, y_smote, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED)
dup_auc = c2st_auc(X_dup, y_dup, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED)

auc_table = pd.DataFrame(
    {
        "task": ["real_vs_smote", "real_vs_duplicated"],
        "mean_auc": [smote_auc["mean"], dup_auc["mean"]],
        "std_auc": [smote_auc["std"], dup_auc["std"]],
        "se_auc": [smote_auc["se"], dup_auc["se"]],
        "ci_low": [smote_auc["ci_low"], dup_auc["ci_low"]],
        "ci_high": [smote_auc["ci_high"], dup_auc["ci_high"]],
        "n_folds": [smote_auc["n_folds"], dup_auc["n_folds"]],
    }
)
display(auc_table)

print(
    f"real-vs-SMOTE: mean AUC={smote_auc['mean']:.6f} +/- {smote_auc['std']:.6f} "
    f"(95% CI for mean [{smote_auc['ci_low']:.6f}, {smote_auc['ci_high']:.6f}])"
)
print(
    f"real-vs-duplicated: mean AUC={dup_auc['mean']:.6f} +/- {dup_auc['std']:.6f} "
    f"(95% CI for mean [{dup_auc['ci_low']:.6f}, {dup_auc['ci_high']:.6f}])"
)

,task,mean_auc,std_auc,se_auc,ci_low,ci_high,n_folds
0,real_vs_smote,0.860849,0.007294,0.001883,0.857158,0.86454,15
1,real_vs_duplicated,0.500000,0.000000,0.000000,0.500000,0.50000,15


real-vs-SMOTE: mean AUC=0.860849 +/- 0.007294 (95% CI for mean [0.857158, 0.864540])
real-vs-duplicated: mean AUC=0.500000 +/- 0.000000 (95% CI for mean [0.500000, 0.500000])


## 3. Permutation importances

In [4]:
smote_importances = c2st_importances(X_smote, y_smote, seed=SEED)
top_importances = smote_importances.head(20).copy()
display(top_importances)

,feature,importance_mean,importance_std
0,TAG_N-2,0.033191,0.004055
1,TAG_N-7,0.020795,0.003114
2,STATUS_BAYAR_N-2,0.018470,0.003291
3,TAG_N-3,0.017293,0.002423
4,STATUS_BAYAR_N-6,0.017047,0.002214
5,TAG_N-8,0.014627,0.002595
6,GGN_N-3,0.014468,0.002638
7,TAG_N-4,0.011284,0.003101
8,GGN_N-4,0.010838,0.002382
9,USAGE_N-4,0.010278,0.003153


## 4. Interpretation

In [5]:
def _auc_label(mean_auc: float) -> str:
    if mean_auc <= 0.55:
        return "indistinguishable"
    if mean_auc <= 0.70:
        return "mildly detectable"
    return "clearly detectable"

smote_label = _auc_label(smote_auc["mean"])
dup_label = _auc_label(dup_auc["mean"])

if smote_auc["mean"] > 0.55:
    importance_sentence = (
        "Because the real-vs-SMOTE AUC is meaningfully above 0.55, the top "
        "permutation importances are interpreted as evidence about where point-level "
        "synthetic detectability concentrates."
    )
else:
    importance_sentence = (
        "Because the real-vs-SMOTE AUC is near the indistinguishable range, the "
        "permutation importances are reported for transparency but should not be "
        "treated as a stable mechanism explanation."
    )

interpretation = f"""
**C2ST result.** The real-vs-SMOTE classifier has mean ROC-AUC {smote_auc['mean']:.4f} (std {smote_auc['std']:.4f}; 95% CI for the mean [{smote_auc['ci_low']:.4f}, {smote_auc['ci_high']:.4f}]), which is **{smote_label}** by the notebook's rule of thumb.

**Sanity floor.** The real-vs-duplicated classifier has mean ROC-AUC {dup_auc['mean']:.4f} (std {dup_auc['std']:.4f}), which is **{dup_label}**. Since duplicated rows are copies of real rows, this checks that the C2ST harness is not spuriously detecting differences when none should exist.

{importance_sentence}

**Scope.** C2ST asks whether complete flattened minority customers are point-level distinguishable from synthetic ones across all selected signals and honest-window lags. It does not test ranking performance, calibration, threshold effects, or causal mechanism by itself. Notebook 02's finding that SMOTE harms PR-AUC on the honest window remains the ranking evidence; this notebook only probes one possible explanation. If C2ST is inconclusive, the mechanism remains open rather than disproven.
"""

display(Markdown(interpretation))


**C2ST result.** The real-vs-SMOTE classifier has mean ROC-AUC 0.8608 (std 0.0073; 95% CI for the mean [0.8572, 0.8645]), which is **clearly detectable** by the notebook's rule of thumb.

**Sanity floor.** The real-vs-duplicated classifier has mean ROC-AUC 0.5000 (std 0.0000), which is **indistinguishable**. Since duplicated rows are copies of real rows, this checks that the C2ST harness is not spuriously detecting differences when none should exist.

Because the real-vs-SMOTE AUC is meaningfully above 0.55, the top permutation importances are interpreted as evidence about where point-level synthetic detectability concentrates.

**Scope.** C2ST asks whether complete flattened minority customers are point-level distinguishable from synthetic ones across all selected signals and honest-window lags. It does not test ranking performance, calibration, threshold effects, or causal mechanism by itself. Notebook 02's finding that SMOTE harms PR-AUC on the honest window remains the ranking evidence; this notebook only probes one possible explanation. If C2ST is inconclusive, the mechanism remains open rather than disproven.


## 5. Verification report

In [6]:
def _df_to_md_table(frame: pd.DataFrame, float_format: str = "{:.6f}") -> str:
    header = "| " + " | ".join([frame.index.name or ""] + list(frame.columns)) + " |"
    sep = "| " + " | ".join(["---"] * (len(frame.columns) + 1)) + " |"
    lines = [header, sep]
    for idx, row in frame.iterrows():
        row_cells = [str(idx)]
        for value in row:
            if isinstance(value, (float, np.floating)):
                row_cells.append(float_format.format(float(value)))
            else:
                row_cells.append(str(value))
        lines.append("| " + " | ".join(row_cells) + " |")
    return "\n".join(lines)


def _present(value: float) -> bool:
    return bool(np.isfinite(value))


smote_balance = float(y_smote.mean())
dup_balance = float(y_dup.mean())
dup_auc_pass = 0.47 <= dup_auc["mean"] <= 0.53
balance_pass = 0.45 <= smote_balance <= 0.55 and 0.45 <= dup_balance <= 0.55
uncertainty_present = all(
    _present(value)
    for value in [smote_auc["mean"], smote_auc["std"], smote_auc["ci_low"], smote_auc["ci_high"]]
)

checks = [
    {
        "check": "real-vs-duplicated C2ST AUC in [0.47, 0.53]",
        "measured": f"{dup_auc['mean']:.6f}",
        "target": "0.47 <= AUC <= 0.53",
        "result": "PASS" if dup_auc_pass else "FAIL",
    },
    {
        "check": "C2ST tasks are class-balanced within [0.45, 0.55] positive rate",
        "measured": f"real_vs_smote={smote_balance:.6f}, real_vs_duplicated={dup_balance:.6f}",
        "target": "both positive rates in [0.45, 0.55]",
        "result": "PASS" if balance_pass else "FAIL",
    },
    {
        "check": "real-vs-SMOTE AUC and 95% interval are reported",
        "measured": (
            f"mean={smote_auc['mean']:.6f}, std={smote_auc['std']:.6f}, "
            f"ci=[{smote_auc['ci_low']:.6f}, {smote_auc['ci_high']:.6f}]"
        ),
        "target": "reported; no PASS/FAIL on AUC value",
        "result": "PASS" if uncertainty_present else "FAIL",
    },
]
checks_table = pd.DataFrame(checks)
for check in checks:
    print(
        f"[{check['result']}] {check['check']} | measured: {check['measured']} | "
        f"target: {check['target']}"
    )

[PASS] real-vs-duplicated C2ST AUC in [0.47, 0.53] | measured: 0.500000 | target: 0.47 <= AUC <= 0.53
[PASS] C2ST tasks are class-balanced within [0.45, 0.55] positive rate | measured: real_vs_smote=0.500000, real_vs_duplicated=0.500000 | target: both positive rates in [0.45, 0.55]
[PASS] real-vs-SMOTE AUC and 95% interval are reported | measured: mean=0.860849, std=0.007294, ci=[0.857158, 0.864540] | target: reported; no PASS/FAIL on AUC value


In [7]:
c2st_source = inspect.getsource(c2st_module)
file_sha256_12 = hashlib.sha256(RAW_CHURN_PATH.read_bytes()).hexdigest()[:12]

auc_report_table = auc_table.set_index("task")
importance_report_table = top_importances.set_index("feature")
checks_report_table = checks_table.set_index("check")

raw_auc_json = json.dumps(auc_report_table.to_dict(orient="index"), indent=2)
raw_importance_json = json.dumps(top_importances.to_dict(orient="records"), indent=2)
raw_metadata_json = json.dumps(metadata, indent=2)

report_lines = [
    "# 05 ? C2ST Mechanism Probe Verification Report",
    "",
    "## Environment",
    f"- Python: {platform.python_version()}",
    f"- numpy: {np.__version__}",
    f"- pandas: {pd.__version__}",
    f"- scikit-learn: {sklearn.__version__}",
    f"- imbalanced-learn: {imblearn.__version__}",
    f"- scipy: {scipy.__version__}",
    f"- matplotlib: {matplotlib.__version__}",
    f"- Seed: {SEED}",
    f"- Raw data SHA256(first 12): {file_sha256_12}",
    "",
    "## CV config",
    f"- n_splits: {N_SPLITS}",
    f"- n_repeats: {N_REPEATS}",
    f"- n_folds per task: {smote_auc['n_folds']}",
    "- Classifier: src.models.factory.make_gbm",
    "",
    "## Group sizes and task balance",
    f"- n_real: {metadata['real_vs_smote']['n_real']:,}",
    f"- n_smote_synthetic_total: {metadata['real_vs_smote']['n_smote_synthetic_total']:,}",
    f"- n_smote_synthetic_used: {metadata['real_vs_smote']['n_smote_synthetic_used']:,}",
    f"- n_duplicated: {metadata['real_vs_duplicated']['n_duplicated']:,}",
    f"- n_dropped_missing before C2ST construction: {metadata['real_vs_smote']['n_dropped_missing']}",
    f"- real-vs-SMOTE positive_rate: {smote_balance:.6f}",
    f"- real-vs-duplicated positive_rate: {dup_balance:.6f}",
    "",
    "Raw metadata JSON:",
    "```json",
    raw_metadata_json,
    "```",
    "",
    "## C2ST AUC table (full precision)",
    "",
    _df_to_md_table(auc_report_table),
    "",
    "Raw JSON:",
    "```json",
    raw_auc_json,
    "```",
    "",
    "## Top permutation importances (real-vs-SMOTE)",
    "",
    _df_to_md_table(importance_report_table),
    "",
    "Raw JSON:",
    "```json",
    raw_importance_json,
    "```",
    "",
    "## Interpretation",
    interpretation.strip(),
    "",
    "## Inline source",
    "",
    "### src/evaluation/c2st.py",
    "```python",
    c2st_source,
    "```",
    "",
    "## Self-run acceptance checks",
    _df_to_md_table(checks_report_table, float_format="{}"),
    "",
]

report_text = "\n".join(report_lines)
report_path = OUTPUT_DIR / "05_c2st_verification.md"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)

# 05 ? C2ST Mechanism Probe Verification Report

## Environment
- Python: 3.12.3
- numpy: 2.4.6
- pandas: 3.0.3
- scikit-learn: 1.9.0
- imbalanced-learn: 0.14.2
- scipy: 1.18.0
- matplotlib: 3.11.0
- Seed: 42
- Raw data SHA256(first 12): c04f3a55aa4f

## CV config
- n_splits: 5
- n_repeats: 3
- n_folds per task: 15
- Classifier: src.models.factory.make_gbm

## Group sizes and task balance
- n_real: 3,804
- n_smote_synthetic_total: 92,198
- n_smote_synthetic_used: 3,804
- n_duplicated: 3,804
- n_dropped_missing before C2ST construction: 1
- real-vs-SMOTE positive_rate: 0.500000
- real-vs-duplicated positive_rate: 0.500000

Raw metadata JSON:
```json
{
  "real_vs_smote": {
    "task": "real_vs_smote",
    "n_real": 3804,
    "n_smote_synthetic_total": 92198,
    "n_smote_synthetic_used": 3804,
    "n_features": 40,
    "n_dropped_missing": 1,
    "positive_rate": 0.5,
    "smote_k_neighbors": 5,
    "months": [
      "N-11",
      "N-10",
      "N-9",
      "N-8",
      "N-7",
      "N-6